# Prepare modelling sample

Первые три шага практической части:
1. Зафиксировать final sample: `MM`-опционы, `market_price = SETTLEPRICE`, `underlying = UNDERLYING_CLOSE`.
2. Посчитать базовые признаки: `DTE`, `moneyness`, `log_moneyness`, `year/month`, regime labels, historical volatility windows.
3. Сделать pragmatic quality filter для modelling.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
project_root = Path('/Users/maria/Desktop/Code/HSE/COURSEBOOK')
input_path = project_root / 'data/final/imoex_mm_options_with_underlying_2024_2026.parquet'
output_dir = project_root / 'data/final'
output_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_parquet(input_path)
raw['TRADEDATE'] = pd.to_datetime(raw['TRADEDATE'], errors='coerce').dt.normalize()
raw['expiry_date'] = pd.to_datetime(raw['expiry_date'], errors='coerce').dt.normalize()
raw['strike'] = pd.to_numeric(raw['strike'], errors='coerce')
raw['SETTLEPRICE'] = pd.to_numeric(raw['SETTLEPRICE'], errors='coerce')
raw['UNDERLYING_CLOSE'] = pd.to_numeric(raw['UNDERLYING_CLOSE'], errors='coerce')
raw['DTE_DAYS'] = pd.to_numeric(raw['DTE_DAYS'], errors='coerce')

df = raw.copy()
df['market_price'] = df['SETTLEPRICE']
df['underlying_price'] = df['UNDERLYING_CLOSE']

print('Input shape:', df.shape)
print('Columns:', list(df.columns))

Input shape: (19574, 22)
Columns: ['SECID', 'TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'SETTLEPRICE', 'VALUE', 'VOLUME', 'OPENPOSITION', 'NUMTRADES', 'option_type', 'strike', 'expiry_date', 'UNDERLYING_CLOSE', 'UNDERLYING_OPEN', 'UNDERLYING_HIGH', 'UNDERLYING_LOW', 'UNDERLYING_VOLUME', 'DTE_DAYS', 'market_price', 'underlying_price']


## 1. Final sample

In [3]:
final_sample_summary = {
    'rows': len(df),
    'unique_secids': int(df['SECID'].nunique()),
    'trade_date_min': df['TRADEDATE'].min().date(),
    'trade_date_max': df['TRADEDATE'].max().date(),
    'expiry_min': df['expiry_date'].min().date(),
    'expiry_max': df['expiry_date'].max().date(),
    'option_types': df['option_type'].value_counts(dropna=False).to_dict(),
}
final_sample_summary

{'rows': 19574,
 'unique_secids': 92,
 'trade_date_min': datetime.date(2024, 1, 3),
 'trade_date_max': datetime.date(2026, 5, 19),
 'expiry_min': datetime.date(2026, 4, 16),
 'expiry_max': datetime.date(2027, 6, 17),
 'option_types': {'C': 13217, 'P': 6357}}

## 2. Feature engineering

In [4]:
underlying_daily = (
    df[['TRADEDATE', 'underlying_price']]
    .drop_duplicates()
    .sort_values('TRADEDATE')
    .reset_index(drop=True)
)

underlying_daily['ret_1d'] = np.log(underlying_daily['underlying_price']).diff()
underlying_daily['ret_21d'] = np.log(
    underlying_daily['underlying_price'] / underlying_daily['underlying_price'].shift(21)
)
underlying_daily['hv_21d'] = underlying_daily['ret_1d'].rolling(21).std() * np.sqrt(252)
underlying_daily['hv_63d'] = underlying_daily['ret_1d'].rolling(63).std() * np.sqrt(252)

vol_q1 = underlying_daily['hv_21d'].quantile(1 / 3)
vol_q2 = underlying_daily['hv_21d'].quantile(2 / 3)

underlying_daily['vol_regime'] = pd.cut(
    underlying_daily['hv_21d'],
    bins=[-np.inf, vol_q1, vol_q2, np.inf],
    labels=['low_vol', 'mid_vol', 'high_vol'],
)
underlying_daily['trend_regime'] = np.where(
    underlying_daily['ret_21d'] > 0,
    'up',
    'down',
)
underlying_daily['regime_label'] = np.where(
    underlying_daily['vol_regime'].isna(),
    pd.NA,
    underlying_daily['vol_regime'].astype(str) + '_' + underlying_daily['trend_regime'],
)

df = df.merge(
    underlying_daily[
        ['TRADEDATE', 'ret_1d', 'ret_21d', 'hv_21d', 'hv_63d', 'vol_regime', 'trend_regime', 'regime_label']
    ],
    on='TRADEDATE',
    how='left',
)

df['moneyness'] = df['underlying_price'] / df['strike']
df['log_moneyness'] = np.log(df['moneyness'])
df['year'] = df['TRADEDATE'].dt.year
df['month'] = df['TRADEDATE'].dt.month

feature_cols = [
    'SECID',
    'TRADEDATE',
    'expiry_date',
    'option_type',
    'strike',
    'market_price',
    'underlying_price',
    'DTE_DAYS',
    'moneyness',
    'log_moneyness',
    'ret_1d',
    'ret_21d',
    'hv_21d',
    'hv_63d',
    'vol_regime',
    'trend_regime',
    'regime_label',
    'year',
    'month',
]
df[feature_cols].head()

,SECID,TRADEDATE,expiry_date,option_type,strike,market_price,underlying_price,DTE_DAYS,moneyness,log_moneyness,ret_1d,ret_21d,hv_21d,hv_63d,vol_regime,trend_regime,regime_label,year,month
0,MM2500BR6,2024-01-03,2026-06-18,P,2500.0,37.55,3130.23,897,1.252092,0.224816,NaN,NaN,NaN,NaN,NaN,down,<NA>,2024,1
1,MM2550BR6,2024-01-03,2026-06-18,P,2550.0,40.45,3130.23,897,1.227541,0.205013,NaN,NaN,NaN,NaN,NaN,down,<NA>,2024,1
2,MM2600BF6,2024-01-03,2026-06-18,C,2600.0,1523.90,3130.23,897,1.203935,0.185595,NaN,NaN,NaN,NaN,NaN,down,<NA>,2024,1
3,MM2600BR6,2024-01-03,2026-06-18,P,2600.0,43.65,3130.23,897,1.203935,0.185595,NaN,NaN,NaN,NaN,NaN,down,<NA>,2024,1
4,MM2650BF6,2024-01-03,2026-06-18,C,2650.0,1477.35,3130.23,897,1.181219,0.166547,NaN,NaN,NaN,NaN,NaN,down,<NA>,2024,1


In [5]:
feature_summary = {
    'moneyness_range': (float(df['moneyness'].min()), float(df['moneyness'].max())),
    'log_moneyness_range': (float(df['log_moneyness'].min()), float(df['log_moneyness'].max())),
    'dte_range': (float(df['DTE_DAYS'].min()), float(df['DTE_DAYS'].max())),
    'hv_21d_non_null_share': float(df['hv_21d'].notna().mean()),
    'hv_63d_non_null_share': float(df['hv_63d'].notna().mean()),
    'regime_non_null_share': float(df['regime_label'].notna().mean()),
    'regime_counts': df['regime_label'].value_counts(dropna=False).to_dict(),
}
feature_summary

{'moneyness_range': (0.6890289855072464, 1.7412684210526315),
 'log_moneyness_range': (-0.3724719399023412, 0.5546138252677676),
 'dte_range': (0.0, 1189.0),
 'hv_21d_non_null_share': 0.9773679370593644,
 'hv_63d_non_null_share': 0.9295494022683151,
 'regime_non_null_share': 0.9773679370593644,
 'regime_counts': {'low_vol_up': 4507,
  'mid_vol_down': 3697,
  'high_vol_down': 3521,
  'low_vol_down': 2897,
  'high_vol_up': 2305,
  'mid_vol_up': 2204,
  <NA>: 443}}

## 3. Quality filter for modelling

In [6]:
filter_checks = {
    'bad_dte_nonpositive': int((df['DTE_DAYS'] <= 0).sum()),
    'bad_dte_too_long': int((df['DTE_DAYS'] > 3 * 365).sum()),
    'bad_nonpositive_market_price': int((df['market_price'] <= 0).sum()),
    'bad_missing_or_nonpositive_underlying': int(df['underlying_price'].isna().sum() + (df['underlying_price'] <= 0).sum()),
    'bad_missing_or_nonpositive_strike': int(df['strike'].isna().sum() + (df['strike'] <= 0).sum()),
    'bad_option_type': int((~df['option_type'].isin(['C', 'P'])).sum()),
    'bad_moneyness_outside_soft_bounds': int((~df['moneyness'].between(0.7, 1.8)).sum()),
    'bad_price_above_strike_plus_underlying': int((df['market_price'] > (df['strike'] + df['underlying_price'])).sum()),
}
pd.Series(filter_checks).sort_values(ascending=False)

bad_dte_too_long                          236
bad_dte_nonpositive                         9
bad_nonpositive_market_price                6
bad_moneyness_outside_soft_bounds           2
bad_missing_or_nonpositive_underlying       0
bad_missing_or_nonpositive_strike           0
bad_option_type                             0
bad_price_above_strike_plus_underlying      0
dtype: int64

In [7]:
modelling_mask = (
    (df['DTE_DAYS'] > 0)
    & (df['DTE_DAYS'] <= 3 * 365)
    & (df['market_price'] > 0)
    & (df['underlying_price'] > 0)
    & (df['strike'] > 0)
    & (df['option_type'].isin(['C', 'P']))
    & (df['moneyness'].between(0.7, 1.8))
    & (df['market_price'] <= (df['strike'] + df['underlying_price']))
)

modelling_df = df.loc[modelling_mask].copy()
modelling_df = modelling_df.sort_values(['TRADEDATE', 'expiry_date', 'strike', 'option_type', 'SECID']).reset_index(drop=True)

print('Original rows:', len(df))
print('Filtered rows:', len(modelling_df))
print('Removed rows:', len(df) - len(modelling_df))
print('Share kept:', round(len(modelling_df) / len(df), 4))

Original rows: 19574
Filtered rows: 19327
Removed rows: 247
Share kept: 0.9874


In [8]:
final_summary = {
    'rows': len(modelling_df),
    'unique_secids': int(modelling_df['SECID'].nunique()),
    'unique_trade_dates': int(modelling_df['TRADEDATE'].nunique()),
    'date_min': modelling_df['TRADEDATE'].min().date(),
    'date_max': modelling_df['TRADEDATE'].max().date(),
    'rows_by_year': modelling_df['year'].value_counts().sort_index().to_dict(),
    'regime_non_null_share': float(modelling_df['regime_label'].notna().mean()),
    'hv_21d_non_null_share': float(modelling_df['hv_21d'].notna().mean()),
    'hv_63d_non_null_share': float(modelling_df['hv_63d'].notna().mean()),
}
final_summary

{'rows': 19327,
 'unique_secids': 92,
 'unique_trade_dates': 604,
 'date_min': datetime.date(2024, 1, 3),
 'date_max': datetime.date(2026, 5, 19),
 'rows_by_year': {2024: 6683, 2025: 7359, 2026: 5285},
 'regime_non_null_share': 0.9792518238733379,
 'hv_21d_non_null_share': 0.9792518238733379,
 'hv_63d_non_null_share': 0.9353236405029234}

In [9]:
parquet_path = output_dir / 'imoex_mm_options_modelling_sample.parquet'
csv_path = output_dir / 'imoex_mm_options_modelling_sample.csv'

modelling_df.to_parquet(parquet_path, index=False)
modelling_df.to_csv(csv_path, index=False)

print('Saved:', parquet_path)
print('Saved:', csv_path)

Saved: /Users/maria/Desktop/Code/HSE/COURSEBOOK/data/final/imoex_mm_options_modelling_sample.parquet
Saved: /Users/maria/Desktop/Code/HSE/COURSEBOOK/data/final/imoex_mm_options_modelling_sample.csv


## Короткий вывод

- Рабочий sample зафиксирован: `MM`-опционы, `market_price = SETTLEPRICE`, `underlying = UNDERLYING_CLOSE`.
- Базовые признаки для modelling посчитаны.
- Фильтр мягкий: он вычищает только явный мусор и почти не режет выборку.
- Полученный `modelling_df` можно использовать как стартовую таблицу для `Black-76`, `Black-Scholes`, IV и дальнейшего анализа ошибок.